# Lesson 10 | Why do spikes need a queue?

A shared engine may already be busy when new spikes arrive. Today asks:
> **When events temporarily arrive faster than they can be consumed, how do we preserve order without silently losing them?**

Primary concept: a **bounded First-In, First-Out queue (FIFO)**.


## 1. Concept ledger

**Known:** a spike is a discrete event; a shared engine may be busy.

**New:** event queue, FIFO, full/empty, and backpressure.

**Preview:** the next lesson explains how a source spike finds downstream synapses.


## 2. What is an event?

Here a spike becomes a tiny **event record**. The minimal record contains only `source_id`: which neuron spiked. Later systems may add time or metadata, but this lesson does not need them.


## 3. What does FIFO guarantee?

**First-In, First-Out (FIFO):** the earliest arriving event leaves first.

If events enter as `2, 5, 7`, normal pop order must also be `2, 5, 7`. FIFO is about buffering and order, not the biological meaning of a spike.


## 4. Why does a bounded queue need backpressure?

A real hardware queue has finite capacity. When full, the producer must not pretend the next event was accepted.

**Backpressure** is a control condition telling the producer “not ready yet; hold/wait.” In the Python model, `accepted=False` represents that condition.

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 840 300" role="img" aria-label="l10-flow" style="width:100%;height:auto">
<defs><marker id="l10-flow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 Z" fill="#506070"/></marker></defs>
<rect x="25" y="100" width="200" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="125.0" y="135.0" text-anchor="middle" font-family="sans-serif" font-size="15">producer / pending</text>
<rect x="325" y="100" width="170" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="410.0" y="135.0" text-anchor="middle" font-family="sans-serif" font-size="15">bounded FIFO</text>
<rect x="595" y="100" width="180" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="685.0" y="135.0" text-anchor="middle" font-family="sans-serif" font-size="15">consumer</text>
<path d="M225 130 H325" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l10-flow)"/>
<text x="270" y="118" text-anchor="middle" font-family="sans-serif" font-size="14">push</text>
<path d="M495 130 H595" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l10-flow)"/>
<text x="545" y="118" text-anchor="middle" font-family="sans-serif" font-size="14">pop</text>
<path d="M410 100 V40 H125 V100" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l10-flow)"/>
<text x="270" y="30" text-anchor="middle" font-family="sans-serif" font-size="14">full / retry later</text>
</svg>

In this teaching API, `accepted=False` means **this push did not happen**. The caller still owns the event and must not forget it; after space opens, the same event should be retried. It is not permission to drop the spike.


## 5. Run: a queue with capacity 3

Predict what happens to the fourth event, `9`, then run.

### Before running: full, empty, and rejected events

The **producer** submits and the **consumer** removes events. A capacity-three FIFO has zero to three entries: zero is empty, three is full. `deque.append` adds at the tail; `popleft` removes the head.

The example retains a rejected event in `pending`, consumes one entry to make room, then retries the same event. Remove it from pending only after acceptance. Predict whether consumption remains `2,5,7,9`.


In [ ]:
from collections import deque

capacity = 3
q = deque()
pending = deque()
consumed = []

def show(action):
    print(f'{action:18s} queue={list(q)} full={len(q) == capacity} empty={len(q) == 0}')

for event in [2, 5, 7, 9]:
    if len(q) < capacity:
        q.append(event)
        show(f'accepted spike {event}')
    else:
        pending.append(event)
        show(f'blocked spike {event}')

while q or pending:
    if q:
        event = q.popleft()
        consumed.append(event)
        show(f'consumed spike {event}')
    if pending and len(q) < capacity:
        event = pending[0]
        q.append(event)
        pending.popleft()
        show(f'retried spike {event}')

print('consumed order:', consumed)
print('pending:', list(pending))


## 6. Observe

The first three events enter the FIFO. Event 9 remains pending. Consuming 2 makes room to retry 9, leaving `[5,7,9]`. Final consumption is `[2,5,7,9]` and pending is empty. Backpressure delays submission while preserving content and order.


## 7. Try It: slow consumer

Change capacity to 2. Predict which push blocks first. Then try “push two → pop one → push one” and observe backpressure being released.


## 8. Homework

Open:

[Lesson 10 Exercise: Bounded FIFO and backpressure](../../exercises/en/10_spike_fifo_backpressure.ipynb)

The workbook focuses on FIFO ordering, full/empty behavior, and retry semantics for a blocked event. Concrete grader vectors are not displayed directly in the Exercise Notebook.

## 9. AI Task

Ask AI for the shortest capacity-2 operation sequence that visits empty, non-empty, full, and backpressured states. Require expected queue contents before code.


## 10. Human Check

Without AI, explain why FIFO preserves order, what full/empty mean, why backpressure is safer than overwriting an old spike, and why a queue transports events rather than changing neuron semantics.


## 11. Engineering Handoff

This lesson aligns conceptually with `MOD-005 spike_fifo`, `IF-SPIKE-QUEUE`, T-007 FIFO ordering, and T-008 FIFO backpressure. It is not yet the formal RTL FIFO.


## 12. Project Trace

- Lesson: `LSN-010`
- Mapping: `RMD-007A / RMD-009` teaching precursor
- Module context: `MOD-005`
- Test context: `T-007 / T-008`


## 13. Exit Ticket

You can explain bounded-FIFO ordering and backpressure, and recognize why a full queue must explicitly delay or reject a new event rather than silently overwrite data.
